In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# ציון אפשרויות

<details>
<summary><b>גרסאות חבילות</b></summary>

הקוד בדף זה פותח עם הדרישות הבאות.
אנחנו ממליצים להשתמש בגרסאות אלה או חדשות יותר.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
אפשר להשתמש באפשרויות כדי להתאים אישית את הפרימיטיבים Estimator ו-Sampler. חלק זה מתמקד בדרכים לציין אפשרויות פרימיטיב של Qiskit Runtime. בעוד שהממשק של מתודת `run()` של הפרימיטיבים משותף לכל המימושים, האפשרויות שלהם אינן כך. עיינו בהפניות ה-API המתאימות לקבלת מידע על האפשרויות של [`qiskit.primitives`](https://docs.quantum.ibm.com/api/qiskit/primitives#primitives) ו-[`qiskit_aer.primitives`](https://qiskit.github.io/qiskit-aer/apidocs/aer_primitives.html).

הערות לגבי ציון אפשרויות בפרימיטיבים:

- ל-`SamplerV2` ול-`EstimatorV2` יש מחלקות אפשרויות נפרדות. אפשר לראות את האפשרויות הזמינות ולעדכן ערכי אפשרויות במהלך אתחול הפרימיטיב או אחריו.
- השתמשו במתודה `update()` כדי להחיל שינויים על מאפיין `options`.
- אם לא מציינים ערך עבור אפשרות, היא מקבלת ערך מיוחד של `Unset` וברירות המחדל של השרת משמשות.
- מאפיין `options` הוא מסוג Python `dataclass`. אפשר להשתמש במתודה המובנית `asdict` כדי להמיר אותו למילון.

<span id="pass-options"></span>
## הגדרת אפשרויות פרימיטיב
אפשר להגדיר אפשרויות בעת אתחול הפרימיטיב, לאחר אתחולו, או במתודה `run()`. עיינו בסעיף [כללי עדיפות](runtime-options-overview#options-precedence) כדי להבין מה קורה כשאותה אפשרות מצוינת במספר מקומות.

### אתחול הפרימיטיב
אפשר להעביר מופע של מחלקת האפשרויות או מילון בעת אתחול פרימיטיב, שאז עושה עותק של אותן אפשרויות. לפיכך, שינוי המילון המקורי או מופע האפשרויות אינו משפיע על האפשרויות השייכות לפרימיטיבים.

#### מחלקת אפשרויות
בעת יצירת מופע של מחלקת `EstimatorV2` או `SamplerV2`, אפשר להעביר מופע של מחלקת האפשרויות. אפשרויות אלה יוחלו לאחר מכן כשמשתמשים ב-`run()` לביצוע החישוב. ציינו את האפשרויות בפורמט זה: `options.option.sub-option.sub-sub-option = choice`. לדוגמה: `options.dynamical_decoupling.enable = True`

דוגמה:

ל-`SamplerV2` ול-`EstimatorV2` יש מחלקות אפשרויות נפרדות ([`EstimatorOptions`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options) ו-[`SamplerOptions`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-sampler-options)).

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime.options import EstimatorOptions

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

options = EstimatorOptions(
    resilience_level=2,
    resilience={"zne_mitigation": True, "zne": {"noise_factors": [1, 3, 5]}},
)

# or...
options = EstimatorOptions()
options.resilience_level = 2
options.resilience.zne_mitigation = True
options.resilience.zne.noise_factors = [1, 3, 5]

estimator = Estimator(mode=backend, options=options)

#### מילון
אפשר לציין אפשרויות כמילון בעת אתחול הפרימיטיב.

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during primitive initialization
estimator = Estimator(
    backend,
    options={
        "resilience_level": 2,
        "resilience": {
            "zne_mitigation": True,
            "zne": {"noise_factors": [1, 3, 5]},
        },
    },
)

### עדכון אפשרויות לאחר אתחול
אפשר לציין את האפשרויות בפורמט זה: `primitive.options.option.sub-option.sub-sub-option = choice` כדי לנצל השלמה אוטומטית, או להשתמש במתודה `update()` לעדכונים מרוכזים.

מחלקות האפשרויות של `SamplerV2` ו-`EstimatorV2` ([`EstimatorOptions`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options) ו-[`SamplerOptions`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-sampler-options)) אינן צריכות להיות מאותחלות אם מגדירים אפשרויות לאחר אתחול הפרימיטיב.

In [3]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(mode=backend)

# Setting options after primitive initialization
# This uses auto-complete.
estimator.options.default_shots = 4000
# This does bulk update.
estimator.options.update(
    default_shots=4000, resilience={"zne_mitigation": True}
)

<span id="run-method"></span>
### מתודת Run()
הערכים היחידים שאפשר להעביר ל-`run()` הם אלה המוגדרים בממשק. כלומר, `shots` עבור Sampler ו-`precision` עבור Estimator. זה מחליף כל ערך שהוגדר עבור `default_shots` או `default_precision` עבור הריצה הנוכחית.

In [4]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)

sampler = Sampler(mode=backend)
# Default shots to use if not specified in run()
sampler.options.default_shots = 500
# Sample two circuits at 128 shots each.
sampler.run([transpiled1, transpiled2], shots=128)

# Sample two circuits with different numbers of shots.
# 100 shots is used for transpiled1 and 200 for transpiled.
sampler.run([(transpiled1, None, 100), (transpiled2, None, 200)])

<RuntimeJobV2('d5k96cn853es738djikg', 'sampler')>

### Special cases

#### Resilience level (Estimator only)

The resilience level is not actually an option that directly impacts the primitive query, but specifies a base set of curated options to build off of. In general, level 0 turns off all error mitigation, level 1 turns on options for measurement error mitigation, and level 2 turns on options for gate and measurement error mitigation.

Any options you manually specify in addition to the resilience level are applied on top of the base set of options defined by the resilience level. Therefore, in principle, you could set the resilience level to 1, but then turn off measurement mitigation, although this is not advised.

In the following example, setting the resilience level to 0 initially turns off `zne_mitigation`, but `estimator.options.resilience.zne_mitigation = True` overrides the relevant setup from `estimator.options.resilience_level = 0`.

In [5]:
from qiskit_ibm_runtime import EstimatorV2, QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = EstimatorV2(backend)

estimator.options.default_shots = 100
estimator.options.resilience_level = 0
estimator.options.resilience.zne_mitigation = True

### מקרים מיוחדים
#### רמת חוסן (Estimator בלבד)
רמת החוסן אינה למעשה אפשרות שמשפיעה ישירות על שאילתת הפרימיטיב, אלא מציינת קבוצת בסיס של אפשרויות מאורגנות לבנות עליהן. באופן כללי, רמה 0 מכבה את כל הפחתת השגיאות, רמה 1 מפעילה אפשרויות להפחתת שגיאות מדידה, ורמה 2 מפעילה אפשרויות להפחתת שגיאות שערים ומדידה.

כל אפשרות שמציינים ידנית בנוסף לרמת החוסן מוחלת על גבי קבוצת האפשרויות הבסיסית שהוגדרה על ידי רמת החוסן. לפיכך, עקרונית אפשר להגדיר את רמת החוסן ל-1, אך לכבות הפחתת שגיאות מדידה, אם כי אין לעשות זאת.

בדוגמה הבאה, הגדרת רמת החוסן ל-0 מכבה תחילה את `zne_mitigation`, אך `estimator.options.resilience.zne_mitigation = True` דורסת את ההגדרה הרלוונטית של `estimator.options.resilience_level = 0`.

In [6]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)


# Setting shots during primitive initialization
sampler = Sampler(mode=backend, options={"default_shots": 4096})

# Setting options after primitive initialization
# This uses auto-complete.
sampler.options.default_shots = 2000

# This does bulk update.  The value for default_shots is overridden if you specify shots with run() or in the PUB.
sampler.options.update(
    default_shots=1024, dynamical_decoupling={"sequence_type": "XpXm"}
)

# Sample two circuits at 128 shots each.
sampler.run([transpiled1, transpiled2], shots=128)

<RuntimeJobV2('d5k96icjt3vs73ds5t0g', 'sampler')>

#### Shots (Sampler בלבד)
מתודת `SamplerV2.run` מקבלת שני ארגומנטים: רשימת PUBים, שכל אחד מהם יכול לציין ערך shots ספציפי ל-PUB, וארגומנט מילת מפתח shots. ערכי ה-shots הללו הם חלק מממשק ביצוע ה-Sampler, ואינם תלויים באפשרויות ה-Sampler של Runtime. הם קודמים לכל ערך שצוין כאפשרויות כדי לציית להפשטת ה-Sampler.

עם זאת, אם `shots` לא צוין על ידי אף PUB או בארגומנט מילת המפתח run (או אם כולם הם `None`), אזי ערך ה-shots מהאפשרויות משמש, ובמיוחד `default_shots`.

לסיכום, זהו סדר העדיפויות לציון shots ב-Sampler, עבור כל PUB מסוים:

1. אם ה-PUB מציין shots, השתמשו בערך זה.
2. אם ארגומנט מילת המפתח `shots` מצוין ב-`run`, השתמשו בערך זה.
3. אם `num_randomizations` ו-`shots_per_randomization` מצוינים כאפשרויות `twirling`, ה-shots הם מכפלת הערכים הללו.
3. אם `sampler.options.default_shots` מצוין, השתמשו בערך זה.

לפיכך, אם shots מצוינים בכל המקומות האפשריים, זה עם העדיפות הגבוהה ביותר (shots המצוינים ב-PUB) משמש.

#### Precision (Estimator בלבד)
Precision דומה ל-shots, המתואר בסעיף הקודם, פרט לכך שאפשרויות ה-Estimator מכילות גם `default_shots` וגם `default_precision`. בנוסף, מכיוון שסיבוב שערים מופעל כברירת מחדל, מכפלת `num_randomizations` ו-`shots_per_randomization` קודמת לשתי אפשרויות אלה.

ספציפית, עבור כל Estimator PUB מסוים:

1. אם ה-PUB מציין precision, השתמשו בערך זה.
2. אם ארגומנט מילת המפתח precision מצוין ב-`run`, השתמשו בערך זה.
2. אם `num_randomizations` ו-`shots_per_randomization` מצוינים כ-[`twirling` options](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options) (מופעל כברירת מחדל), השתמשו במכפלתם לשליטה בכמות הנתונים.
3. אם `estimator.options.default_shots` מצוין, השתמשו בערך זה לשליטה בכמות הנתונים.
4. אם `estimator.options.default_precision` מצוין, השתמשו בערך זה.

לדוגמה, אם precision מצוין בכל ארבעת המקומות, זה עם העדיפות הגבוהה ביותר (precision המצוין ב-PUB) משמש.

> **Note:** Precision משתנה ביחס הפוך לשימוש. כלומר, ככל ש-precision נמוך יותר, כך נדרש יותר זמן QPU להריץ.

## אפשרויות בשימוש נפוץ
קיימות אפשרויות רבות, אך הבאות הן הנפוצות ביותר:

<span id="shots"></span>
### Shots
עבור חלק מהאלגוריתמים, הגדרת מספר ספציפי של shots היא חלק מרכזי בשגרותיהם. Shots (או precision) יכולים להיות מצוינים במספר מקומות. סדר העדיפויות הוא כדלקמן:

עבור כל Sampler PUB:

1. shots בעלי ערך שלם הכלולים ב-PUB
2. הערך `run(...,shots=val)`
3. הערך `options.default_shots`

עבור כל Estimator PUB:

1. precision בעל ערך עשרוני הכלול ב-PUB
2. הערך `run(...,precision=val)`
3. הערך `options.default_shots`
4. הערך `options.default_precision`

דוגמה:

In [7]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(mode=backend)

estimator.options.max_execution_time = 2500

<span id="no-error-mitigation"></span>
## Turn off all error mitigation and error suppression

You can turn off all error mitigation and suppression if you are, for example, doing research on your own mitigation techniques. To accomplish this, for EstimatorV2, set `resilience_level = 0`. For SamplerV2, no changes are necessary because no error mitigation or suppression options are enabled by default.

Example:

Turn off all error mitigation and suppression in Estimator.

In [8]:
from qiskit_ibm_runtime import EstimatorV2 as Estimator, QiskitRuntimeService

# Define the service.  This allows you to access IBM QPU.
service = QiskitRuntimeService()

# Get a backend
backend = service.least_busy(operational=True, simulator=False)

# Define Estimator
estimator = Estimator(backend)

options = estimator.options

# Turn off all error mitigation and suppression
options.resilience_level = 0

### זמן ביצוע מרבי
זמן הביצוע המרבי (`max_execution_time`) מגביל כמה זמן משימה יכולה לרוץ. אם משימה חורגת ממגבלת זמן זו, היא מבוטלת בכפייה. ערך זה חל על משימות בודדות, בין אם הן פועלות במצב משימה, Session או אצווה.

הערך מוגדר בשניות, בהתבסס על זמן קוונטי (לא זמן שעון קיר), שהוא כמות הזמן שה-QPU מוקדש לעיבוד המשימה שלך. הוא מתעלם בעת שימוש במצב בדיקה מקומי מכיוון שמצב זה אינו משתמש בזמן קוונטי.